# 08a — Prompt iteration: `graph_endpoint_v1` (Graph linker, Method G)

Purpose: sanity-check `GRAPH_ENDPOINT_V1` on the **same 20** Spider train
examples Weeks 6/7 used (`data/processed/prompt_iteration_set.json`,
`seed=42`) — same selection, same sqlglot Tier-1 train gold — so Methods
C/D/G stay directly comparable on identical questions. This is Method G's
endpoint-identification prompt only; the tables actually predicted also
depend on `schema_linking.utils.graph`'s shortest-path/Steiner algorithm,
which is not itself under iteration here (it's pure, already unit-tested in
`tests/test_graph.py`).

**This notebook makes real calls to the Anthropic API** (Haiku 4.5), bounded
by `cost_cap_usd=25.0` — the cumulative cap agreed for Weeks 6+7+8 combined,
shared across `outputs/logs/llm_calls_prompt_iteration.jsonl`. Never run on
Spider **dev** here — dev is reserved for the final reported numbers.

Constraints (locked before running):
- Same 20 `question_id`s as Weeks 6/7 — no resampling.
- Gold is sqlglot-derived Tier-1 **train** gold
  (`data/processed/gold_links_train_mentioned.json`).
- Method G is deterministic (`k_samples=1`, `temperature=0.0`, locked in
  `GraphLinker.__init__`) — no self-consistency sampling, so (like Week 7,
  unlike Week 6) there is no run-to-run noise to average over: identical
  input always gives identical output.
- Do not tune the prompt to fix specific examples — only general rule
  changes are in scope for a `graph_endpoint_v2` draft.
- Failure categories are the fixed vocabulary: Parse error, Hallucinated
  endpoint, Wrong endpoint, Graph disconnect, Column drift, Steiner
  overshoot, Other.


In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))


def load_dotenv(path: Path) -> None:
    # Minimal .env loader (no python-dotenv dependency for one env var).
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in ("'", '"'):
            value = value[1:-1]
        os.environ.setdefault(key, value)


load_dotenv(REPO_ROOT / ".env")

from schema_linking.base import from_predictions_to_dict
from schema_linking.data_loader import load_spider_questions
from schema_linking.evaluator import evaluate
from schema_linking.graph_linker import GraphLinker
from schema_linking.llm_linker import LLMForwardLinker
from schema_linking.schema_parser import load_schemas
from schema_linking.utils.difficulty import difficulty_for_examples
from schema_linking.utils.llm_client import LLMClient
from schema_linking.utils.prompts import (
    FORWARD_V1,
    GRAPH_ENDPOINT_V1,
    GRAPH_ENDPOINT_V2,
    render_schema_block,
)

LOG_PATH = REPO_ROOT / "outputs" / "logs" / "llm_calls_prompt_iteration.jsonl"
SELECTION_PATH = REPO_ROOT / "data" / "processed" / "prompt_iteration_set.json"
GRAPH_FEWSHOT_PATH = REPO_ROOT / "data" / "processed" / "few_shot_examples_graph.json"
FORWARD_FEWSHOT_PATH = REPO_ROOT / "data" / "processed" / "few_shot_examples.json"
GOLD_TRAIN_TIER1_PATH = REPO_ROOT / "data" / "processed" / "gold_links_train_mentioned.json"
TRACE_PATH_V1 = REPO_ROOT / "outputs" / "predictions" / "graph_train_iteration_traces_v1.jsonl"
TRACE_PATH_V2 = REPO_ROOT / "outputs" / "predictions" / "graph_train_iteration_traces_v2.jsonl"

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 160)

## 1. Load train examples, schemas, sqlglot Tier-1 train gold, hardness

In [2]:
train_examples = list(load_spider_questions("train"))
schemas = load_schemas()
hardness = difficulty_for_examples(train_examples)

gold_raw = json.load(GOLD_TRAIN_TIER1_PATH.open())
gold_train = {int(qid): entry for qid, entry in gold_raw.items()}

print(f"{len(train_examples)} train examples, {len(schemas)} schemas, {len(gold_train)} Tier-1 gold entries")

7000 train examples, 166 schemas, 7000 Tier-1 gold entries


## 2. Load graph few-shot examples, enrich with rendered schema blocks

`GraphLinker` expects each few-shot dict to carry its own `schema_block`
(that example's own db, rendered once) — same convention as
`LLMForwardLinker`/`LLMBackwardLinker`. Source: `few_shot_examples_graph.json`
(Week 7's `to_graph_fewshot_examples` reformat of Week 6's forward picks —
a pure key rename, since Tier-1 gold already excludes join-bridge-only
tables — see `src/schema_linking/utils/fewshot.py`).

In [3]:
graph_few_shot = json.load(GRAPH_FEWSHOT_PATH.open())
for ex in graph_few_shot:
    ex["schema_block"] = render_schema_block(schemas[ex["db_id"]])

for ex in graph_few_shot:
    print(f"{ex['pattern']:>12}: qid={ex['question_id']} db={ex['db_id']!r} core_tables={ex['core_tables']} — {ex['question']!r}")

      simple: qid=4913 db='store_product' core_tables=['district'] — 'What is the total number of residents for the districts with the 3 largest areas?'
 multi_table: qid=334 db='product_catalog' core_tables=['Attribute_Definitions', 'Catalog_Contents_Additional_Attributes'] — 'Which attribute definitions have attribute value 0? Give me the attribute name and attribute ID.'


## 3. Reuse the fixed 20-example selection (no resampling)

Same `question_id`s Weeks 6/7 used — `data/processed/prompt_iteration_set.json`,
`seed=42`, 5 easy / 9 medium / 3 hard / 3 extra Spider hardness buckets.

In [4]:
selection_records = json.load(SELECTION_PATH.open())
selected_qids = [r["question_id"] for r in selection_records]
examples_by_qid = {ex.question_id: ex for ex in train_examples}
selected = [examples_by_qid[qid] for qid in selected_qids]

assert len(selected) == 20
print(f"Reusing Weeks 6/7's {len(selected)} selected examples (same seed=42 set, no resampling)")
pd.DataFrame(selection_records)[["question_id", "db_id", "hardness", "question"]]

Reusing Weeks 6/7's 20 selected examples (same seed=42 set, no resampling)


,question_id,db_id,hardness,question
0,1550,customers_and_invoices,easy,Count the number of customers who have an account.
1,4052,student_1,easy,Find the last names of teachers teaching in classroom 109.
2,6623,driving_school,easy,What are the ids of all vehicles?
3,235,musical,easy,Count the number of actors.
4,452,allergy_1,easy,How many animal type allergies exist?
5,324,product_catalog,medium,Which catalog content has the highest height? Give me the catalog entry name.
6,963,medicine_enzyme_interaction,medium,What is the id and trade name of the medicines can interact with at least 3 enzymes?
7,3128,assets_maintenance,medium,How many assets does each third party company supply? List the count and the company id.
8,2054,party_people,medium,Which minister left office the latest?
9,5546,products_gen_characteristics,medium,"What is the color code and description of the product named ""chervil""?"


## 4. Cost cap

Shared `cost_cap_usd = 25.0` across Weeks 6+7+8 (raised from Week 7's
`cumulative_cap + 3` scheme now that all three prompt-iteration phases share
one log file) — well above what a 20-example, single-call-per-example,
deterministic method needs.

In [5]:
already_logged = 0.0
if LOG_PATH.exists():
    with LOG_PATH.open() as f:
        for line in f:
            line = line.strip()
            if line:
                already_logged += json.loads(line)["cost_usd"]

COST_CAP_USD = 25.0
print(f"Already logged in {LOG_PATH.name} (Weeks 6+7+8 combined): ${already_logged:.5f}")
print(f"This notebook's cost_cap_usd: ${COST_CAP_USD:.5f}")

Already logged in llm_calls_prompt_iteration.jsonl (Weeks 6+7+8 combined): $0.67252
This notebook's cost_cap_usd: $25.00000


## 5. Run `graph_endpoint_v1` on the 20 examples (k=1, T=0)

Looping `GraphLinker._predict_with_raw_text` (not `predict_one`) so each
example's inspection block can print the LLM's **raw**, pre-filter response
alongside the post-graph-algorithm final prediction — `_predict_with_raw_text`
is the same private method `predict_all` uses internally to build its
mandatory trace file; here we build the equivalent trace lines ourselves
(`TRACE_PATH_V1`) rather than calling `predict_all` a second time, to avoid
doubling the API spend on this iteration set.

In [6]:
llm_client_v1 = LLMClient(
    model="claude-haiku-4-5-20251001",
    temperature=0.0,
    max_tokens=1024,
    log_path=LOG_PATH,
    cost_cap_usd=COST_CAP_USD,
)
linker_v1 = GraphLinker(
    llm_client=llm_client_v1,
    prompt=GRAPH_ENDPOINT_V1,
    few_shot=graph_few_shot,
    schemas=schemas,
    trace_path=TRACE_PATH_V1,
)

predictions_v1 = {}
raw_texts_v1 = {}
trace_lines_v1 = []
for i, ex in enumerate(selected):
    schema = schemas[ex.db_id]
    pred, raw_text = linker_v1._predict_with_raw_text(ex, schema)
    predictions_v1[ex.question_id] = pred
    raw_texts_v1[ex.question_id] = raw_text
    gold_entry = gold_train[ex.question_id]
    trace_lines_v1.append({
        "qid": ex.question_id, "db_id": ex.db_id, "question": ex.question,
        "llm_raw": raw_text,
        "endpoints_resolved": list(pred.extra["llm_endpoints_resolved"]),
        "graph_result": list(pred.extra["graph_path_or_subgraph"]),
        "final_tables": list(pred.tables),
        "final_columns": [list(c) for c in pred.columns],
        "failure": pred.extra["failure"],
    })
    print("=" * 100)
    print(f"[{i + 1:>2}/20] qid={ex.question_id:<5} db={ex.db_id:<28} hardness={hardness[ex.question_id]:<6} "
          f"cost=${pred.extra['cost_usd']:.5f}")
    print(f"Question           : {ex.question}")
    print(f"Raw LLM response   : {raw_text!r}")
    print(f"Resolved endpoints : {pred.extra['llm_endpoints_resolved']} (n_hallucinated={pred.extra['n_endpoints_hallucinated']})")
    print(f"Graph path/subgraph: {pred.extra['graph_path_or_subgraph']}")
    print(f"Final tables       : {list(pred.tables)}")
    print(f"Final columns      : {[list(c) for c in pred.columns]} (dropped={pred.extra['columns_dropped_off_path']})")
    print(f"Gold Tier-1        : tables={gold_entry['tables']} columns={gold_entry['columns']}")
    print(f"Failure            : {pred.extra['failure']}")

TRACE_PATH_V1.parent.mkdir(parents=True, exist_ok=True)
with TRACE_PATH_V1.open("w", encoding="utf-8") as f:
    for line in trace_lines_v1:
        f.write(json.dumps(line) + "\n")

v1_total_cost = sum(p.extra["cost_usd"] for p in predictions_v1.values())
print(f"\ngraph_endpoint_v1 total cost: ${v1_total_cost:.5f}")
print(f"Trace written to {TRACE_PATH_V1}")

[ 1/20] qid=1550  db=customers_and_invoices       hardness=easy   cost=$0.00274
Question           : Count the number of customers who have an account.
Raw LLM response   : '```json\n{\n  "core_tables": ["Customers", "Accounts"],\n  "columns": [["Customers", "customer_id"], ["Accounts", "customer_id"]]\n}\n```'
Resolved endpoints : ['Customers', 'Accounts'] (n_hallucinated=0)
Graph path/subgraph: ('Customers', 'Accounts')
Final tables       : ['Customers', 'Accounts']
Final columns      : [['Customers', 'customer_id'], ['Accounts', 'customer_id']] (dropped=0)
Gold Tier-1        : tables=['Accounts'] columns=[['Accounts', 'customer_id']]
Failure            : None


[ 2/20] qid=4052  db=student_1                    hardness=easy   cost=$0.00133
Question           : Find the last names of teachers teaching in classroom 109.
Raw LLM response   : '```json\n{\n  "core_tables": ["teachers"],\n  "columns": [["teachers", "LastName"], ["teachers", "Classroom"]]\n}\n```'
Resolved endpoints : ['teachers'] (n_hallucinated=0)
Graph path/subgraph: ('teachers',)
Final tables       : ['teachers']
Final columns      : [['teachers', 'LastName'], ['teachers', 'Classroom']] (dropped=0)
Gold Tier-1        : tables=['teachers'] columns=[['teachers', 'Classroom'], ['teachers', 'LastName']]
Failure            : None


[ 3/20] qid=6623  db=driving_school               hardness=easy   cost=$0.00215
Question           : What are the ids of all vehicles?
Raw LLM response   : '```json\n{\n  "core_tables": ["Vehicles"],\n  "columns": [["Vehicles", "vehicle_id"]]\n}\n```'
Resolved endpoints : ['Vehicles'] (n_hallucinated=0)
Graph path/subgraph: ('Vehicles',)
Final tables       : ['Vehicles']
Final columns      : [['Vehicles', 'vehicle_id']] (dropped=0)
Gold Tier-1        : tables=['Vehicles'] columns=[['Vehicles', 'vehicle_id']]
Failure            : None


[ 4/20] qid=235   db=musical                      hardness=easy   cost=$0.00140
Question           : Count the number of actors.
Raw LLM response   : '```json\n{\n  "core_tables": ["actor"],\n  "columns": [["actor", "Actor_ID"]]\n}\n```'
Resolved endpoints : ['actor'] (n_hallucinated=0)
Graph path/subgraph: ('actor',)
Final tables       : ['actor']
Final columns      : [['actor', 'Actor_ID']] (dropped=0)
Gold Tier-1        : tables=['actor'] columns=[]
Failure            : None


[ 5/20] qid=452   db=allergy_1                    hardness=easy   cost=$0.00153
Question           : How many animal type allergies exist?
Raw LLM response   : '```json\n{\n  "core_tables": ["Allergy_Type"],\n  "columns": [["Allergy_Type", "AllergyType"]]\n}\n```'
Resolved endpoints : ['Allergy_Type'] (n_hallucinated=0)
Graph path/subgraph: ('Allergy_Type',)
Final tables       : ['Allergy_Type']
Final columns      : [['Allergy_Type', 'AllergyType']] (dropped=0)
Gold Tier-1        : tables=['Allergy_Type'] columns=[['Allergy_Type', 'AllergyType']]
Failure            : None


...

[16/20] qid=4610  db=entertainment_awards         hardness=hard   cost=$0.00170
Question           : Show the names of festivals that have nominated artworks of type "Program Talent Show".
Raw LLM response   : '```json\n{\n  "core_tables": ["festival_detail", "artwork", "nomination"],\n  "columns": [\n    ["festival_detail", "Festival_Name"],\n    ["artwork", "Type"],\n    ["nomination", "Festival_ID"],\n    ["nomination", "Artwork_ID"]\n  ]\n}\n```'
Resolved endpoints : ['festival_detail', 'artwork', 'nomination'] (n_hallucinated=0)
Graph path/subgraph: ('artwork', 'festival_detail', 'nomination')
Final tables       : ['artwork', 'festival_detail', 'nomination']
Final columns      : [['festival_detail', 'Festival_Name'], ['artwork', 'Type'], ['nomination', 'Festival_ID'], ['nomination', 'Artwork_ID']] (dropped=0)
Gold Tier-1        : tables=['artwork', 'festival_detail'] columns=[['artwork', 'Type'], ['festival_detail', 'Festival_Name']]
Failure            : None


[17/20] qid=3385  db=sports_competition           hardness=hard   cost=$0.00189
Question           : which countries did participated in both Friendly and Tournament type competitions.
Raw LLM response   : '```json\n{\n  "core_tables": ["competition", "competition_result"],\n  "columns": [["competition", "Country"], ["competition", "Competition_type"], ["competition_result", "Competition_ID"]]\n}\n```'
Resolved endpoints : ['competition', 'competition_result'] (n_hallucinated=0)
Graph path/subgraph: ('competition', 'competition_result')
Final tables       : ['competition', 'competition_result']
Final columns      : [['competition', 'Country'], ['competition', 'Competition_type'], ['competition_result', 'Competition_ID']] (dropped=0)
Gold Tier-1        : tables=['competition'] columns=[['competition', 'Competition_type'], ['competition', 'Country']]
Failure            : None


[18/20] qid=5877  db=tracking_share_transactions  hardness=extra  cost=$0.00218
Question           : Show the description of the transaction type that occurs most frequently.
Raw LLM response   : '```json\n{\n  "core_tables": ["Transactions", "Ref_Transaction_Types"],\n  "columns": [\n    ["Transactions", "transaction_type_code"],\n    ["Ref_Transaction_Types", "transaction_type_code"],\n    ["Ref_Transaction_Types", "transaction_type_description"]\n  ]\n}\n```'
Resolved endpoints : ['Transactions', 'Ref_Transaction_Types'] (n_hallucinated=0)
Graph path/subgraph: ('Transactions', 'Ref_Transaction_Types')
Final tables       : ['Transactions', 'Ref_Transaction_Types']
Final columns      : [['Transactions', 'transaction_type_code'], ['Ref_Transaction_Types', 'transaction_type_code'], ['Ref_Transaction_Types', 'transaction_type_description']] (dropped=0)
Gold Tier-1        : tables=['Ref_Transaction_Types'] columns=[['Ref_Transaction_Types', 'transaction_type_code'], ['Ref_Transaction_Type

[19/20] qid=5794  db=customer_complaints          hardness=extra  cost=$0.00217
Question           : Find the last name of the staff member who processed the complaint of the cheapest product.
Raw LLM response   : '```json\n{\n  "core_tables": ["Complaints", "Products", "Staff"],\n  "columns": [\n    ["Products", "product_id"],\n    ["Products", "product_price"],\n    ["Complaints", "product_id"],\n    ["Complaints", "staff_id"],\n    ["Staff", "staff_id"],\n    ["Staff", "last_name"]\n  ]\n}\n```'
Resolved endpoints : ['Complaints', 'Products', 'Staff'] (n_hallucinated=0)
Graph path/subgraph: ('Complaints', 'Products', 'Staff')
Final tables       : ['Complaints', 'Products', 'Staff']
Final columns      : [['Products', 'product_id'], ['Products', 'product_price'], ['Complaints', 'product_id'], ['Complaints', 'staff_id'], ['Staff', 'staff_id'], ['Staff', 'last_name']] (dropped=0)
Gold Tier-1        : tables=['Products', 'Staff'] columns=[['Products', 'product_price'], ['Staff', 'last_na

[20/20] qid=3641  db=baseball_1                   hardness=extra  cost=$0.00665
Question           : In 2014, what are the id and rank of the team that has the largest average number of attendance?
Raw LLM response   : '```json\n{\n  "core_tables": ["team"],\n  "columns": [["team", "year"], ["team", "team_id"], ["team", "rank"], ["team", "attendance"]]\n}\n```'
Resolved endpoints : ['team'] (n_hallucinated=0)
Graph path/subgraph: ('team',)
Final tables       : ['team']
Final columns      : [['team', 'year'], ['team', 'team_id'], ['team', 'rank'], ['team', 'attendance']] (dropped=0)
Gold Tier-1        : tables=['home_game', 'team'] columns=[['home_game', 'attendance'], ['home_game', 'team_id'], ['home_game', 'year'], ['team', 'rank'], ['team', 'team_id']]
Failure            : None

graph_endpoint_v1 total cost: $0.04317
Trace written to /Users/mac/Documents/Masters BHT 2023-2025/Semester 6 2026 april-sep/code/outputs/predictions/graph_train_iteration_traces_v1.jsonl


## 6. Per-query F1 vs sqlglot Tier-1 train gold

`mean_f1 = (table_f1 + column_f1) / 2`, matching Weeks 6/7's selection-rule
convention.

In [7]:
def per_query_table(predictions: dict, method_name: str) -> pd.DataFrame:
    pred_dict = from_predictions_to_dict(predictions)
    gold_subset = {qid: gold_train[qid] for qid in pred_dict}
    result = evaluate(
        predictions=pred_dict, gold=gold_subset, schemas=schemas, hardness=hardness,
        method_name=method_name, tier_name="tier1",
    )
    per_query = result.per_query.copy()
    per_query["mean_f1"] = (per_query["table_f1"] + per_query["column_f1"]) / 2
    return per_query.sort_values("mean_f1").reset_index(drop=True)


per_query_v1 = per_query_table(predictions_v1, "graph_endpoint_v1")
per_query_v1[["question_id", "db_id", "hardness", "table_f1", "column_f1", "mean_f1"]]

,question_id,db_id,hardness,table_f1,column_f1,mean_f1
0,235,musical,easy,1.000000,0.000000,0.500000
1,3641,baseball_1,extra,0.666667,0.444444,0.555556
2,5794,customer_complaints,extra,0.800000,0.500000,0.650000
3,963,medicine_enzyme_interaction,medium,0.666667,0.666667,0.666667
4,1550,customers_and_invoices,easy,0.666667,0.666667,0.666667
5,3128,assets_maintenance,medium,0.666667,0.666667,0.666667
6,4610,entertainment_awards,hard,0.800000,0.666667,0.733333
7,5877,tracking_share_transactions,extra,0.666667,0.800000,0.733333
8,3385,sports_competition,hard,0.666667,0.800000,0.733333
9,2891,icfp_1,hard,0.800000,1.000000,0.900000


In [8]:
n_imperfect_v1 = int((per_query_v1["mean_f1"] < 1.0).sum())
print(f"{n_imperfect_v1}/20 examples are not a perfect mean_f1=1.0")
print(f"Mean table_f1={per_query_v1['table_f1'].mean():.4f}  "
      f"column_f1={per_query_v1['column_f1'].mean():.4f}  "
      f"mean_f1={per_query_v1['mean_f1'].mean():.4f}")

11/20 examples are not a perfect mean_f1=1.0
Mean table_f1=0.8700  column_f1=0.8034  mean_f1=0.8367


## 7. Failure categorisation (manual, against the fixed 7-category vocabulary)

All 11 imperfect examples, hand-categorised against `graph_endpoint_v1`'s
actual raw response, resolved endpoints, graph result, and gold (§5 above
has the full per-example detail; summarised here).

| qid | db | Category | Root cause |
| --- | --- | --- | --- |
| 235 | musical | **Other** | Table correct (`actor`), but the LLM added a "representative" column (`Actor_ID`) for a bare `COUNT(*)` — gold has zero columns. Same pattern `forward_v1` showed on this *exact* qid in Week 6 (`docs/decisions.md`) — a cross-method, model-level tendency, not specific to the graph prompt. |
| 3641 | baseball_1 | **Wrong endpoint** | Single endpoint `team` only; gold needs `team` + `home_game` (`attendance`/`year` live on `home_game`, not `team` — though `team` does have its own same-named `attendance`/`rank` columns, so nothing gets dropped, it's just the wrong source table). Same qid, same root cause (`LLMForwardLinker`, `LLMBackwardLinker` — see Weeks 6/7) already flagged as a genuine domain-reasoning gap, not a prompt-wording issue. |
| 5794 | customer_complaints | **Steiner overshoot** | LLM named `Complaints` as a 3rd core table alongside `Products`/`Staff`, when `Complaints` is exactly the join-bridge table rule 2 says to leave for automatic graph discovery. Because it was named as an *endpoint* rather than left implicit, its own (irrelevant) columns leak straight through un-dropped. |
| 963 | medicine_enzyme_interaction | **Wrong endpoint** | `medicine_enzyme_interaction` named as a 2nd core table; it's a pure join-bridge (`HAVING COUNT(*) >= 3`, no output column) — rule 2 violation at the 2-endpoint level (not literally a 3-endpoint "Steiner" case, but the same underlying mistake). |
| 1550 | customers_and_invoices | **Wrong endpoint** | `Customers` named as a 2nd core table; gold SQL never joins to it at all (`SELECT count(DISTINCT customer_id) FROM Accounts`). Identical qid, identical over-inclusion `forward_v1` showed in Week 6 ("topically plausible but unreferenced"). |
| 3128 | assets_maintenance | **Wrong endpoint** | `Assets` named alongside `Third_Party_Companies`; Tier-1 gold only credits `Third_Party_Companies`. (Note: unlike 1550/963, `Assets` is arguably genuinely needed by the real SQL to *count* assets — this may partly be a Tier-1 "mentioned-only" gold-scoring artifact rather than a pure over-inclusion error; flagged, not corrected, per Week 7's precedent for not re-litigating the gold extractor here.) |
| 4610 | entertainment_awards | **Steiner overshoot** | `nomination` (pure join-bridge, no output column) named as a 3rd core table alongside `festival_detail`/`artwork` — textbook case of the category as defined. |
| 5877 | tracking_share_transactions | **Wrong endpoint** | `Transactions` named as a 2nd core table; Tier-1 gold only credits `Ref_Transaction_Types`. |
| 3385 | sports_competition | **Wrong endpoint** | `competition_result` named as a 2nd core table; gold SQL is a same-table `INTERSECT` over `competition` alone. Identical qid, identical failure `forward_v1` showed in Week 6 (its dominant worst-5 pattern, "included join-only column/table"). |
| 5546 | products_gen_characteristics | **Other** | Tables correct; `color_code` attributed to `Ref_Colors` instead of `Products` (gold's actual SQL selects the denormalised FK-side copy on `Products`) — a column-to-table mix-up between two adjacent, both-real tables, not a hallucination, not an off-path drop. |
| 2891 | icfp_1 | **Other** (not a true failure — see note) | Endpoints `Authors`/`Papers` exactly match gold; `shortest_path_tables` correctly auto-adds the connecting `Authorship` bridge table since `Authors` and `Papers` aren't directly FK-linked. This is *correct, intended* graph-algorithm behaviour — Method G's design always includes the necessary join path — but Tier-1 ("Mentioned") gold never credits a bridge table, so it costs `table_f1` regardless. Structurally identical to Week 7's "Excess join column" caveat for Method D: an evaluation-tier artifact of scoring a join-completing method against mention-only gold, not a method defect. |

**None of the 20 examples hit Parse error, Hallucinated endpoint, Graph
disconnect, or Column drift** — Haiku 4.5 never hallucinated a table/column
name against these schemas, and every FK graph in this sample was connected
enough that shortest-path/Steiner never needed the "add the terminal alone,
disconnected" fallback.

## 8. STOP — tally and decision

| Category | Count | qids |
| --- | --- | --- |
| Wrong endpoint | 6 | 3641, 963, 1550, 3128, 5877, 3385 |
| Other | 3 | 235, 5546, 2891 |
| Steiner overshoot | 2 | 5794, 4610 |

**Decision rule:** ≥3 failures in a fixable category → draft `graph_endpoint_v2`.

**6/11 (55%) share "Wrong endpoint"**, clearing the threshold. Is it
fixable? The category splits into two general (non-schema-specific)
sub-patterns, both already addressable by generalising existing rules
(mirroring `forward_v2`'s Week-6 rule-3 rewrite, extended from columns to
tables):

1. **Topically-plausible-but-unreferenced second table** (1550, 3128, 3385)
   — rule 1 currently says nothing about a core table needing to be
   *actually needed by the SQL*, only "return between 1 and 3."
2. **Join-bridge table misnamed as a core table** (963, 5877, and the two
   "Steiner overshoot" cases 5794/4610) — rule 2 already forbids this in
   principle, but gives no concrete test for "join-only," so the model
   evidently can't reliably self-apply it.

**-> Drafting `graph_endpoint_v2`** (rules 1 and 2 rewritten with an
explicit "needed by the SQL" criterion and a concrete join-only test; rules
3-5 and the user template/output schema unchanged — see
`src/schema_linking/utils/prompts.py`). qid=235's and 5546's "Other" and
2891's evaluation-artifact case don't meet the ≥3 threshold on their own and
aren't obviously prompt-fixable (235/2891 mirror already-known cross-method
or scoring-tier issues; 5546 is a one-off column/table mix-up).

In [9]:
print("--- graph_endpoint_v1 system ---")
print(GRAPH_ENDPOINT_V1.system)
print()
print("--- graph_endpoint_v2 system ---")
print(GRAPH_ENDPOINT_V2.system)

--- graph_endpoint_v1 system ---
You identify the 1 to 3 core tables and the specific columns needed from a database schema to answer a natural-language question.

Rules:
1. Return between 1 and 3 core tables. Prefer fewer.
2. Do NOT include tables that only exist to join others — those will be added automatically by a graph algorithm.
3. For columns, return only those that would appear in SELECT, WHERE, GROUP BY, HAVING, or ORDER BY of the target SQL. Not join columns.
4. Every table and column must be from the provided schema.
5. Output ONLY a JSON object with keys "core_tables" (list of 1-3 strings) and "columns" (list of [table_name, column_name] pairs).

--- graph_endpoint_v2 system ---
You identify the 1 to 3 core tables and the specific columns needed from a database schema to answer a natural-language question.

Rules:
1. Return between 1 and 3 core tables — tables the target SQL actually needs to compute the answer. Do NOT include a table just because it is topically related t

## 9. Re-run the same 20 examples with `graph_endpoint_v2`

Same examples, same few-shot, same model/temperature(0.0)/k_samples(1) —
only the prompt version differs.

In [10]:
llm_client_v2 = LLMClient(
    model="claude-haiku-4-5-20251001",
    temperature=0.0,
    max_tokens=1024,
    log_path=LOG_PATH,
    cost_cap_usd=COST_CAP_USD,
)
linker_v2 = GraphLinker(
    llm_client=llm_client_v2,
    prompt=GRAPH_ENDPOINT_V2,
    few_shot=graph_few_shot,
    schemas=schemas,
    trace_path=TRACE_PATH_V2,
)

predictions_v2 = {}
raw_texts_v2 = {}
trace_lines_v2 = []
for i, ex in enumerate(selected):
    schema = schemas[ex.db_id]
    pred, raw_text = linker_v2._predict_with_raw_text(ex, schema)
    predictions_v2[ex.question_id] = pred
    raw_texts_v2[ex.question_id] = raw_text
    trace_lines_v2.append({
        "qid": ex.question_id, "db_id": ex.db_id, "question": ex.question,
        "llm_raw": raw_text,
        "endpoints_resolved": list(pred.extra["llm_endpoints_resolved"]),
        "graph_result": list(pred.extra["graph_path_or_subgraph"]),
        "final_tables": list(pred.tables),
        "final_columns": [list(c) for c in pred.columns],
        "failure": pred.extra["failure"],
    })
    print(f"[{i + 1:>2}/20] qid={ex.question_id:<5} db={ex.db_id:<28} cost=${pred.extra['cost_usd']:.5f} "
          f"endpoints={pred.extra['llm_endpoints_resolved']} final_tables={list(pred.tables)}")

TRACE_PATH_V2.parent.mkdir(parents=True, exist_ok=True)
with TRACE_PATH_V2.open("w", encoding="utf-8") as f:
    for line in trace_lines_v2:
        f.write(json.dumps(line) + "\n")

v2_total_cost = sum(p.extra["cost_usd"] for p in predictions_v2.values())
print(f"\ngraph_endpoint_v2 total cost: ${v2_total_cost:.5f}")

[ 1/20] qid=1550  db=customers_and_invoices       cost=$0.00285 endpoints=['Customers', 'Accounts'] final_tables=['Customers', 'Accounts']


[ 2/20] qid=4052  db=student_1                    cost=$0.00145 endpoints=['teachers'] final_tables=['teachers']


[ 3/20] qid=6623  db=driving_school               cost=$0.00227 endpoints=['Vehicles'] final_tables=['Vehicles']


[ 4/20] qid=235   db=musical                      cost=$0.00152 endpoints=['actor'] final_tables=['actor']


[ 5/20] qid=452   db=allergy_1                    cost=$0.00164 endpoints=['Allergy_Type'] final_tables=['Allergy_Type']


...

[16/20] qid=4610  db=entertainment_awards         cost=$0.00182 endpoints=['festival_detail', 'artwork', 'nomination'] final_tables=['artwork', 'festival_detail', 'nomination']


[17/20] qid=3385  db=sports_competition           cost=$0.00194 endpoints=['competition'] final_tables=['competition']


[18/20] qid=5877  db=tracking_share_transactions  cost=$0.00230 endpoints=['Transactions', 'Ref_Transaction_Types'] final_tables=['Transactions', 'Ref_Transaction_Types']


[19/20] qid=5794  db=customer_complaints          cost=$0.00229 endpoints=['Complaints', 'Products', 'Staff'] final_tables=['Complaints', 'Products', 'Staff']


[20/20] qid=3641  db=baseball_1                   cost=$0.00677 endpoints=['team'] final_tables=['team']

graph_endpoint_v2 total cost: $0.04537


In [11]:
per_query_v2 = per_query_table(predictions_v2, "graph_endpoint_v2")
per_query_v2[["question_id", "db_id", "hardness", "table_f1", "column_f1", "mean_f1"]]

,question_id,db_id,hardness,table_f1,column_f1,mean_f1
0,235,musical,easy,1.000000,0.000000,0.500000
1,3641,baseball_1,extra,0.666667,0.444444,0.555556
2,5794,customer_complaints,extra,0.800000,0.500000,0.650000
3,963,medicine_enzyme_interaction,medium,0.666667,0.666667,0.666667
4,1550,customers_and_invoices,easy,0.666667,0.666667,0.666667
5,3128,assets_maintenance,medium,0.666667,0.666667,0.666667
6,4610,entertainment_awards,hard,0.800000,0.666667,0.733333
7,5877,tracking_share_transactions,extra,0.666667,0.800000,0.733333
8,5546,products_gen_characteristics,medium,1.000000,0.666667,0.833333
9,2891,icfp_1,hard,0.800000,1.000000,0.900000


In [12]:
comparison = pd.DataFrame({
    "v1_mean_table_f1": [per_query_v1["table_f1"].mean()],
    "v1_mean_column_f1": [per_query_v1["column_f1"].mean()],
    "v1_mean_f1": [per_query_v1["mean_f1"].mean()],
    "v2_mean_table_f1": [per_query_v2["table_f1"].mean()],
    "v2_mean_column_f1": [per_query_v2["column_f1"].mean()],
    "v2_mean_f1": [per_query_v2["mean_f1"].mean()],
})
comparison

,v1_mean_table_f1,v1_mean_column_f1,v1_mean_f1,v2_mean_table_f1,v2_mean_column_f1,v2_mean_f1
0,0.87,0.803413,0.836706,0.886667,0.803889,0.845278


In [13]:
diff = pd.DataFrame({
    "question_id": selected_qids,
    "v1_mean_f1": [per_query_v1.set_index("question_id").loc[qid, "mean_f1"] for qid in selected_qids],
    "v2_mean_f1": [per_query_v2.set_index("question_id").loc[qid, "mean_f1"] for qid in selected_qids],
})
diff["delta"] = diff["v2_mean_f1"] - diff["v1_mean_f1"]
diff[diff["delta"] != 0]

,question_id,v1_mean_f1,v2_mean_f1,delta
9,5546,0.928571,0.833333,-0.095238
16,3385,0.733333,1.000000,0.266667


## 10. Comparison and final decision

**`v1_mean_f1` vs `v2_mean_f1`** (table above): `v2` is marginally ahead
(0.8367 -> 0.8453, `+0.0086`) — smaller than the weight of a single query
moving between the two runs (see the per-query delta table above, which is
exactly what happened: only 2 of the 20 queries changed at all):

- **qid=3385 improved** (0.733 -> 1.0): `graph_endpoint_v2` correctly
  dropped `competition_result`, predicting `competition` alone — exactly
  the target pattern, and the *only* one of the 6 "Wrong endpoint" cases
  the rewrite actually fixed.
- **qid=5546 regressed** (0.929 -> 0.833): under `v1`, the model listed
  *both* `Products.color_code` and `Ref_Colors.color_code` among its raw
  columns (redundant, but the `Products` copy happens to be the gold one);
  under `v2`, tightened rule-1/rule-3 wording ("only if the SQL actually
  needs it") pushed the model to drop the seemingly-redundant duplicate —
  but it kept the wrong copy (`Ref_Colors.color_code`) and dropped the
  gold one (`Products.color_code`). This regression has nothing to do with
  the endpoint-naming pattern `v2` was designed to fix — it's a side effect
  of the same wording change pressuring column selection in an unrelated,
  unhelpful direction.
- **1550, 963, 3128, 5877 — unchanged.** All four other "topically
  plausible" / "join-bridge-named-as-core" cases that motivated the rewrite
  reproduce *identically* under `v2` (byte-identical raw responses for
  1550/963/5877; 3128 differs only in table order, not content). The rule
  change demonstrably did not generalise past the one case it happened to
  fix.

**This is the same finding Week 6 made about `forward_v2`**: a
well-motivated, general rule rewrite that fixes the specific case that
inspired it, but does not reproducibly fix the *pattern* — 5/6 "Wrong
endpoint" cases are completely unchanged under `v2`, and the tiny aggregate
gain comes bundled with a new, unrelated regression.

**Decision: lock `graph_endpoint_v1`.** `graph_endpoint_v2` stays registered
in `prompts.py` (useful for future ablation, e.g. with an explicit few-shot
counter-example rather than wording alone) but is not the frozen version.
Simplicity wins a tie that isn't reliably a win.

## 11. Bonus comparison — how often do G's endpoints agree with C's predicted table set?

Subset check: is `graph_endpoint_v1`'s **resolved endpoint set** (the raw
core tables the LLM named, before the graph algorithm adds any bridge
tables) a subset of `forward_v1`'s (Method C, `k_samples=3`,
`temperature=0.3`) predicted table set, on the same 20 questions? Re-running
Method C fresh here (same prompt/few-shot as Weeks 6/7) rather than reusing
old logged predictions, so the comparison is against a live run.

In [14]:
forward_few_shot = json.load(FORWARD_FEWSHOT_PATH.open())
for ex in forward_few_shot:
    ex["schema_block"] = render_schema_block(schemas[ex["db_id"]])

llm_client_c = LLMClient(
    model="claude-haiku-4-5-20251001",
    temperature=0.3,
    max_tokens=1024,
    log_path=LOG_PATH,
    cost_cap_usd=COST_CAP_USD,
)
linker_c = LLMForwardLinker(
    llm_client=llm_client_c,
    prompt=FORWARD_V1,
    few_shot=forward_few_shot,
    k_samples=3,
    aggregation="union",
    extra_metadata={"phase": "graph_prompt_iteration", "prompt_version": FORWARD_V1.version, "purpose": "bonus_comparison"},
)

predictions_c = {}
for ex in selected:
    predictions_c[ex.question_id] = linker_c.predict_one(ex, schemas[ex.db_id])

c_total_cost = sum(p.extra["total_cost_usd"] for p in predictions_c.values())
print(f"Method C (forward_v1) total cost this run: ${c_total_cost:.5f}")

Method C (forward_v1) total cost this run: $0.12504


In [15]:
rows = []
for qid in selected_qids:
    g_endpoints = set(predictions_v1[qid].extra["llm_endpoints_resolved"])
    c_tables = set(predictions_c[qid].tables)
    rows.append({
        "question_id": qid,
        "db_id": examples_by_qid[qid].db_id,
        "G_endpoints": sorted(g_endpoints),
        "C_tables": sorted(c_tables),
        "G_subset_of_C": g_endpoints.issubset(c_tables),
    })
subset_df = pd.DataFrame(rows)
n_subset = int(subset_df["G_subset_of_C"].sum())
print(f"G's resolved endpoints are a subset of C's predicted tables on {n_subset}/20 questions")
subset_df

G's resolved endpoints are a subset of C's predicted tables on 20/20 questions


,question_id,db_id,G_endpoints,C_tables,G_subset_of_C
0,1550,customers_and_invoices,"[Accounts, Customers]","[Accounts, Customers]",True
1,4052,student_1,[teachers],[teachers],True
2,6623,driving_school,[Vehicles],[Vehicles],True
3,235,musical,[actor],[actor],True
4,452,allergy_1,[Allergy_Type],[Allergy_Type],True
5,324,product_catalog,[Catalog_Contents],[Catalog_Contents],True
6,963,medicine_enzyme_interaction,"[medicine, medicine_enzyme_interaction]","[medicine, medicine_enzyme_interaction]",True
7,3128,assets_maintenance,"[Assets, Third_Party_Companies]","[Assets, Third_Party_Companies]",True
8,2054,party_people,[party],[party],True
9,5546,products_gen_characteristics,"[Products, Ref_Colors]","[Products, Ref_Colors]",True


**Result: 20/20 (100%).** `graph_endpoint_v1`'s resolved endpoints were a
subset of `forward_v1`'s predicted table set on every one of the 20
questions — on 19/20 the two sets are exactly equal; the sole difference is
qid=2891, where Method C's self-consistency additionally (and correctly, by
construction of `union` aggregation across `k_samples=3`) surfaces the
`Authorship` bridge table that Method G's graph algorithm adds separately
rather than via the LLM. This is a strong signal that both prompting
strategies elicit the *same* underlying table-selection judgement from
Haiku 4.5 — including the *same* failure cases (1550's `Customers`, 3128's
`Assets`, 3385's `competition_result` — all three qids the "Wrong endpoint"
category flagged for G are ones Week 6 independently flagged for C too).
The graph algorithm's job (auto-adding join-bridge tables) is genuinely
separable from endpoint identification, but endpoint identification itself
doesn't look like a distinct skill between forward-style and graph-endpoint
prompting on this sample.

## 12. Final report

**Locked prompt version: `graph_endpoint_v1`.** Run for real on the same 20
Spider train examples as Weeks 6/7, scored against sqlglot Tier-1 train
gold. `graph_endpoint_v2` was drafted per the decision rule (6/11 imperfect
examples shared "Wrong endpoint", clearing the ≥3 threshold) and re-run on
the identical 20 examples, but only fixed 1 of the 6 motivating cases
(qid=3385) while regressing a different, previously-good case (qid=5546)
via an unrelated column-selection side effect — the net `+0.0086 mean_f1`
movement is not a reproducible improvement (see §10). `graph_endpoint_v1`
is retained as the frozen prompt for the Week 8 dev run.

**Failure category counts** (11/20 imperfect examples under the locked
`graph_endpoint_v1`; 4 categories in the fixed vocabulary — Parse error,
Hallucinated endpoint, Graph disconnect, Column drift — did not occur at
all in this sample):

| Category | Count | qids |
| --- | --- | --- |
| Wrong endpoint | 6 | 3641, 963, 1550, 3128, 5877, 3385 |
| Other | 3 | 235, 5546, 2891 |
| Steiner overshoot | 2 | 5794, 4610 |
| Parse error | 0 | — |
| Hallucinated endpoint | 0 | — |
| Graph disconnect | 0 | — |
| Column drift | 0 | — |

**Total cost spent on this iteration phase** (summed directly from returned
`Prediction.extra["cost_usd"]`/`total_cost_usd`, not the shared log file —
unlike `LLMForwardLinker`/`LLMBackwardLinker`, `GraphLinker.predict_one`'s
call `metadata` is fixed at `{"method", "qid", "db_id"}` with no
`phase`/`prompt_version` tag, so filtering the shared log by phase isn't
available for Method G the way it was for Methods C/D):

In [16]:
phase_total = v1_total_cost + v2_total_cost + c_total_cost
print(f"graph_endpoint_v1 : ${v1_total_cost:.5f}")
print(f"graph_endpoint_v2 : ${v2_total_cost:.5f}")
print(f"forward_v1 (bonus): ${c_total_cost:.5f}")
print(f"Total this phase  : ${phase_total:.5f} (cap was ${COST_CAP_USD:.2f})")

cumulative_after = 0.0
with LOG_PATH.open() as f:
    for line in f:
        line = line.strip()
        if line:
            cumulative_after += json.loads(line)["cost_usd"]
print(f"Cumulative logged in {LOG_PATH.name} after this run (Weeks 6+7+8 combined): ${cumulative_after:.5f}")

graph_endpoint_v1 : $0.04317
graph_endpoint_v2 : $0.04537
forward_v1 (bonus): $0.12504
Total this phase  : $0.21359 (cap was $25.00)
Cumulative logged in llm_calls_prompt_iteration.jsonl after this run (Weeks 6+7+8 combined): $0.88611


**Bonus comparison — how often do G's endpoints agree with C's predicted
table set (subset check)?** **20/20 (100%)** — see §11. G's resolved
endpoints were a subset of C's predicted tables on every question, exactly
equal on 19/20. The two methods' distinct failures (Weeks 6 and this
notebook) largely coincide on the same qids (1550, 3128, 3385), suggesting
the dominant error source is the underlying model's table-selection
judgement rather than either prompt's specific wording.